In [1]:
import numpy as np
data = np.load("/storage_common/nobilm/backmapping_pots_model/datasets/active/with_hs/without_hs/backmapping_dataset.npz")
data = dict(data)
# fix atom_resids by shifting it down by 2 since it was starting at 2 and we like at 0 s.t. avoid any mistake when indexing
new_atom_resids = data['atom_resids'] - 2; new_atom_resids
data['atom_resids'] = new_atom_resids; new_atom_resids
seq = 'SSVYITVELAIAVLAILGNVLVCWAVWLNSNLQNVTNYFVVSLAAADIAVGVLAIPFAITISTGFCAACHGCLFIACFVLVLTQSSIFSLLAIAIDRYIAIRIPLRYNGLVTGTRAKGIIAICWVLSFAIGLTPMLGWNNCGQPKEGKNHSQGCGEGQVACLFEDVVPMNYMVYFNFFACVLVPLLLMLGVYLRIFLAARRQLKQMESQPLPGERARSTLQKEVHAAKSLAIIVGLFALCWLPLHIINCFTFFCPDCSHAPLWLMYLAIVLSHTNSVVNPFIYAYRIREFRQTFRKIIRS'
res_idx = list(set(data['atom_residue_index'].tolist()))
res_idx.sort()
data['res_idx'] = res_idx
[k for k in data.keys()], len(seq)

(['trajectory',
  'atom_resids',
  'atom_names',
  'atom_residue_index',
  'residue_keys',
  'residue_cluster_ids',
  'residue_cluster_counts',
  'frame_indices',
  'frame_state_ids',
  'state_id',
  'sample_id',
  'dihedrals',
  'dihedral_keys',
  'dihedral_atom_indices',
  'dihedral_mask',
  'res_idx'],
 300)

In [3]:
# build the mapping to global_id
l = list(zip(data['res_idx'], data['residue_cluster_counts']))
# build the res_idx_to_glob_cluster_ids:
res_idx_to_glob_cluster_ids = {}
global_id = 0
for res_id, clu_count in l:
    for clu_global_idx in range(clu_count):
        res_idx_to_glob_cluster_ids[(res_id, clu_global_idx)] = global_id
        global_id += 1

# res_idx_to_glob_cluster_ids[(299, 17)] # 1682; 1682*768 = nn.embeddings 1'291'776

In [4]:
# build the actual data
res_idx_and_glob_cluster_id = []
for frame in data['residue_cluster_ids']:
    for_this_frame = []
    for res_idx, cluster_id in enumerate(frame):
        glob_cluster_id = res_idx_to_glob_cluster_ids[(res_idx, cluster_id)]
        # for_this_frame.append((res_idx, glob_cluster_id))
        for_this_frame.append(glob_cluster_id)
    res_idx_and_glob_cluster_id.append(for_this_frame)

In [ ]:
data['res_idx_and_glob_cluster_id_per_frame'] = np.array(res_idx_and_glob_cluster_id) # (49996, 300)
data['atom_idx_and_glob_cluster_id_per_frame'] =  data['res_idx_and_glob_cluster_id_per_frame'][:, new_atom_resids] # (49996, 2338)

In [ ]:
path = '/home/nobilm@usi.ch/ml-simplefold/test_new_data_with_clusters/active_without_hs.npz'
# np.savez_compressed(path, **data)


In [9]:
data_new = dict(np.load(path))
for k,v in data_new.items(): print(k, v.shape)

trajectory (49996, 2338, 3)
atom_resids (2338,)
atom_names (2338,)
atom_residue_index (2338,)
residue_keys (300,)
residue_cluster_ids (49996, 300)
residue_cluster_counts (300,)
frame_indices (49996,)
frame_state_ids (49996,)
state_id ()
sample_id ()
dihedrals (49996, 300, 5)
dihedral_keys (5,)
dihedral_atom_indices (300, 5, 4)
dihedral_mask (300, 5)
res_idx (300,)
res_idx_and_glob_cluster_id_per_frame (49996, 300)
atom_idx_and_glob_cluster_id_per_frame (49996, 2338)
